# NBA Salary Prediction — Preprocessing

Goal: Transform the merged dataset into a model-ready format.
All decisions based on EDA findings.

Key decisions:
- Minimum 10 games played filter — remove small sample size noise
- Drop MP, GS — proxy variables causing multicollinearity
- Drop TRB — redundant with ORB + DRB
- Drop FG% — redundant with eFG%
- Drop ORtg — correlated with eFG% at 0.85
- Log transform salary — confirmed by QQ plots
- Encode position — categorical variable
- Train/test split stratified by salary tier
- Scale numeric features

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pickle

In [4]:
DATA_DIR = Path('../data')

df = pd.read_pickle(DATA_DIR / 'nba_df.pkl')

print(f"Loaded {df.shape[0]} players, {df.shape[1]} columns")
df.head()

Loaded 424 players, 33 columns


,Rk,Player,Age,Team,Pos,G,FG,FGA,FG%,3P,...,STL,BLK,TOV,PF,PTS,ORtg,DRtg,Awards,Player-ID,salary
0,1,Mikal Bridges,28,NYK,SF,82,9.7,19.3,0.500,2.7,...,1.2,0.7,2.2,2.1,23.6,117,118,NaN,bridgmi01,24900000.0
1,2,Josh Hart,29,NYK,SG,77,6.9,13.2,0.525,1.4,...,2.0,0.5,2.7,3.4,18.0,125,112,NaN,hartjo01,19472240.0
2,3,Anthony Edwards,23,MIN,SG,79,12.4,27.7,0.447,5.5,...,1.6,0.9,4.3,2.6,37.4,115,112,MVP-7CPOY-3ASNBA2,edwaran01,45550512.0
3,4,Devin Booker,28,PHO,SG,75,11.6,25.1,0.461,3.2,...,1.2,0.3,3.9,3.5,34.0,119,123,NaN,bookede01,53142264.0
4,5,James Harden,35,LAC,PG,79,9.4,22.9,0.410,4.1,...,2.1,1.0,6.0,2.9,31.8,114,110,MVP-10ASNBA3,hardeja01,39182693.0


## Step 1: Drop unnecessary columns

Based on EDA findings:
- Rk, Player-ID: identifiers, not features
- Awards: 682 nulls, not a performance stat
- TRB: redundant with ORB + DRB (correlation 0.95)
- FG%: redundant with eFG% (correlation 0.87)
- ORtg: correlated with eFG% at 0.85
- FG, FGA, 3P, 3PA, 2P, 2PA, FT, FTA: raw counts — 
  per 100 possession rates already capture this information
- Team: 30 categories, weak salary signal

In [5]:
cols_to_drop = [
    'Rk', 'Player-ID', 'Awards', 'Team',
    'TRB', 'FG%', 'ORtg',
    'FG', 'FGA', '3P', '3PA', '2P', '2PA', 'FT', 'FTA'
]

df = df.drop(columns=cols_to_drop)

print(f"Columns after dropping: {df.shape[1]}")
print(df.columns.tolist())

Columns after dropping: 18
['Player', 'Age', 'Pos', 'G', '3P%', '2P%', 'eFG%', 'FT%', 'ORB', 'DRB', 'AST', 'STL', 'BLK', 'TOV', 'PF', 'PTS', 'DRtg', 'salary']


## Step 2: Minimum games played filter

Remove players with fewer than 10 games played.
Per 100 possession stats are unreliable at small sample sizes —
as confirmed in EDA where James Wiseman (1 game, 57.7 PTS) and 
Alondes Williams (1 game, 60.1 PTS) were clear outliers.

10 games is a conservative threshold that removes noise while 
keeping players who missed significant time due to injury.

In [7]:
before = len(df)
df = df[df['G'] >= 10].reset_index(drop=True)
after = len(df)

print(f"Players before filter: {before}")
print(f"Players after filter:  {after}")
print(f"Players removed:       {before - after}")
print(f"\nRemoved players had G range: 1-9 games")

Players before filter: 411
Players after filter:  411
Players removed:       0

Removed players had G range: 1-9 games


Note: It appears after merging the dataset, we no longer had players with less than 10 games

In [8]:
# Check why we have 411 instead of 424
print(f"Current df shape: {df.shape}")
print(f"\nMissing values per column:")
print(df.isnull().sum()[df.isnull().sum() > 0])
print(f"\nG column range: {df['G'].min()} to {df['G'].max()}")

Current df shape: (411, 18)

Missing values per column:
3P%       11
FT%        1
salary     1
dtype: int64

G column range: 10 to 82


## Step 3: Handle missing values

3P% nulls (11): players who attempted zero 3-pointers — not missing 
data, they simply do not shoot threes. Fill with 0.

FT% nulls (1): player who attempted zero free throws. Fill with 0.

Salary null (1): one player has no salary data — drop this row.
We cannot model salary for a player with no salary target variable.

In [9]:
df['3P%'] = df['3P%'].fillna(0)
df['FT%'] = df['FT%'].fillna(0)


df = df.dropna(subset=['salary']).reset_index(drop=True)

print(f"Players after cleaning: {len(df)}")
print(f"Missing values remaining: {df.isnull().sum().sum()}")

# Check who had null salary
print(f"\nFinal dataset: {df.shape[0]} players, {df.shape[1]} columns")

Players after cleaning: 410
Missing values remaining: 0

Final dataset: 410 players, 18 columns


## Step 4: Log transform salary

QQ plots in EDA confirmed salary is right-skewed.
Log transformation normalizes the distribution and improves 
linear regression performance.

We keep the original salary column for reference and interpretation.
All models will predict log_salary — predictions will be converted 
back to dollars using exp() for interpretability.

In [10]:
# Create log salary target variable
df['log_salary'] = np.log(df['salary'])

print("Salary transformation:")
print(f"  Raw salary   — mean: ${df['salary'].mean():>12,.0f}  std: ${df['salary'].std():>12,.0f}")
print(f"  Log salary   — mean: {df['log_salary'].mean():.3f}  std: {df['log_salary'].std():.3f}")

Salary transformation:
  Raw salary   — mean: $  13,057,036  std: $  14,034,681
  Log salary   — mean: 15.770  std: 1.204


## Step 5: Encode position

Position is a categorical variable with 5 values — PG, SG, SF, PF, C.
We use one-hot encoding with drop_first=True to avoid the dummy 
variable trap in linear regression.

This creates 4 binary columns — the dropped category (C) becomes 
the reference that all other positions are compared against.

In [11]:
df = pd.get_dummies(df, columns=['Pos'], drop_first=True)

print(f"Columns after encoding: {df.shape[1]}")
print(df.columns.tolist())

Columns after encoding: 22
['Player', 'Age', 'G', '3P%', '2P%', 'eFG%', 'FT%', 'ORB', 'DRB', 'AST', 'STL', 'BLK', 'TOV', 'PF', 'PTS', 'DRtg', 'salary', 'log_salary', 'Pos_PF', 'Pos_PG', 'Pos_SF', 'Pos_SG']


## Step 6: Train/test split

80/20 split stratified by salary tier.
Stratification ensures both sets have a proportional mix of 
low, mid, and high salary players — prevents all max contract 
players ending up in one set by chance.

In [12]:
df['salary_tier'] = pd.qcut(df['salary'], q=4, 
                             labels=['low', 'mid', 'high', 'max'])

# Features and target
feature_cols = ['Age', 'G', '3P%', '2P%', 'eFG%', 'FT%', 
                'ORB', 'DRB', 'AST', 'STL', 'BLK', 'TOV', 
                'PF', 'PTS', 'DRtg', 
                'Pos_PF', 'Pos_PG', 'Pos_SF', 'Pos_SG']

X = df[feature_cols]
y = df['log_salary']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, 
    stratify=df['salary_tier']
)

print(f"Training set: {X_train.shape[0]} players")
print(f"Test set:     {X_test.shape[0]} players")
print(f"\nSalary tier distribution in training set:")
print(df.loc[X_train.index, 'salary_tier'].value_counts().sort_index())
print(f"\nSalary tier distribution in test set:")
print(df.loc[X_test.index, 'salary_tier'].value_counts().sort_index())

Training set: 328 players
Test set:     82 players

Salary tier distribution in training set:
salary_tier
low     82
mid     82
high    82
max     82
Name: count, dtype: int64

Salary tier distribution in test set:
salary_tier
low     21
mid     20
high    20
max     21
Name: count, dtype: int64


## Step 7: Scale numeric features

StandardScaler transforms each feature to mean=0, std=1.
Required for Linear Regression — features on different scales 
(Age 20-40 vs DRtg 95-120) would distort coefficients.

Tree-based models (Random Forest, XGBoost) do not require scaling 
but it does not hurt them either — we scale once and use the same 
data for all models.

Critical: fit scaler on training data only, transform both sets.
Fitting on full dataset would leak test set information into training.

In [13]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# Convert back to dataframes to preserve column names
X_train_scaled = pd.DataFrame(X_train_scaled, 
                               columns=feature_cols, 
                               index=X_train.index)
X_test_scaled  = pd.DataFrame(X_test_scaled, 
                               columns=feature_cols, 
                               index=X_test.index)

print("Scaling complete")
print(f"\nTraining set mean after scaling (should be ~0):")
print(X_train_scaled.mean().round(3))
print(f"\nTraining set std after scaling (should be ~1):")
print(X_train_scaled.std().round(3))

Scaling complete

Training set mean after scaling (should be ~0):
Age      -0.0
G         0.0
3P%      -0.0
2P%      -0.0
eFG%      0.0
FT%       0.0
ORB       0.0
DRB      -0.0
AST      -0.0
STL      -0.0
BLK      -0.0
TOV       0.0
PF       -0.0
PTS      -0.0
DRtg     -0.0
Pos_PF   -0.0
Pos_PG   -0.0
Pos_SF   -0.0
Pos_SG   -0.0
dtype: float64

Training set std after scaling (should be ~1):
Age       1.002
G         1.002
3P%       1.002
2P%       1.002
eFG%      1.002
FT%       1.002
ORB       1.002
DRB       1.002
AST       1.002
STL       1.002
BLK       1.002
TOV       1.002
PF        1.002
PTS       1.002
DRtg      1.002
Pos_PF    1.002
Pos_PG    1.002
Pos_SF    1.002
Pos_SG    1.002
dtype: float64


## Step 8: Save processed data

Saving all processed datasets and the scaler as pickle files.
The scaler must be saved to apply identical transformation 
to new data in the Streamlit app.

Also saving player names and actual salaries separately — 
needed for evaluation and the over/underpaid analysis.

In [14]:
X_train_scaled.to_pickle('../data/X_train.pkl')
X_test_scaled.to_pickle('../data/X_test.pkl')
y_train.to_pickle('../data/y_train.pkl')
y_test.to_pickle('../data/y_test.pkl')


player_info = df[['Player', 'salary', 'log_salary', 'Pos_PF', 
                  'Pos_PG', 'Pos_SF', 'Pos_SG']].copy()
player_info.to_pickle('../data/player_info.pkl')


with open('../data/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)


with open('../data/feature_cols.pkl', 'wb') as f:
    pickle.dump(feature_cols, f)

print("Saved:")
print(f"  X_train: {X_train_scaled.shape}")
print(f"  X_test:  {X_test_scaled.shape}")
print(f"  y_train: {y_train.shape}")
print(f"  y_test:  {y_test.shape}")
print(f"  scaler:  fitted on {len(feature_cols)} features")
print(f"  feature_cols: {len(feature_cols)} features")

Saved:
  X_train: (328, 19)
  X_test:  (82, 19)
  y_train: (328,)
  y_test:  (82,)
  scaler:  fitted on 19 features
  feature_cols: 19 features
